In [22]:
from collections import defaultdict
import re

### Kto dotykał najwięcej plików?
Na podstawie: 

`git log --format="===COMMIT===%nAUTHOR:%an" --name-only`

In [4]:
author_files = defaultdict(set)
current_author = None

with open("authors.raw", encoding="utf-8") as f:
    for raw_line in f:
        line = raw_line.strip()
        if not line: continue
        
        if line == "===COMMIT===":
            current_author = None
            continue

        if line.startswith("AUTHOR:"):
            current_author = line.removeprefix("AUTHOR:")
            continue

        if current_author:
            author_files[current_author].add(line)

author_files = sorted(
    author_files.items(),
    key=lambda x: len(x[1]),
    reverse=True
)

print(f"Liczba autorów: {len(author_files)}")

Liczba autorów: 770


In [8]:
for id, (author, files) in enumerate(author_files[:10], 1):
    print(f"{id:>2}. {author:<25} {len(files)}")

 1. Kenneth Reitz             361
 2. Cory Benfield             133
 3. Nate Prewitt              105
 4. Ian Cordasco              100
 5. Ian Stapleton Cordasco    44
 6. Jon Dufresne              38
 7. Seth Michael Larson       34
 8. Kevin Burke               28
 9. Shivaram Lingamneni       23
10. Locker537                 23


### Całkowity churn dla każdego z plików
Na podstawie:

`git log --numstat --format=""`

In [23]:
def normalize_git_filename(name: str) -> str:
    if " => " not in name: return name

    if "{" not in name:
        return name.split(" => ")[1]

    prefix, rest = name.split("{", 1)
    old_new, suffix = rest.split("}", 1)
    _, new = old_new.split(" => ")
    return prefix + new + suffix

In [24]:
files_churn = defaultdict(int)
line_pattern = re.compile(r"^(\d+|-)\s+(\d+|-)\s+(.+)$")

with open("add_del.raw", encoding="utf-8") as f:
    i = 0
    for raw_line in f:
        line = raw_line.strip()
        if not line: continue

        m = line_pattern.match(line)

        if m: 
            adds = m.group(1)
            dels = m.group(2)
            name = m.group(3)

        adds = int(adds) if adds!="-" else 0
        dels = int(dels) if dels!="-" else 0

        churn = adds + dels
        name = normalize_git_filename(name)

        files_churn[name] += churn            

files_churn = sorted(
    files_churn.items(),
    key=lambda x: x[1],
    reverse=True
)

print(f"Liczba plików: {len(files_churn)}")

Liczba plików: 461


In [26]:
churn_sum = sum([churn for _, churn in files_churn])
print(f"Suma churn z wszystkich plików: {churn_sum}")

Suma churn z wszystkich plików: 290692


In [33]:
for id, (name, churn) in enumerate(files_churn[:20], 1):
    print(f"{id:>2}. {name:<45} {churn:,}")

 1. requests/cacert.pem                           23,214
 2. requests/packages/idna/uts46data.py           15,268
 3. ext/requests-logo.ai                          14,432
 4. requests/models.py                            11,079
 5. tests/test_requests.py                        7,684
 6. test_requests.py                              6,490
 7. requests/packages/idna/idnadata.py            6,288
 8. Pipfile.lock                                  5,443
 9. requests/packages/chardet/mbcssm.py           5,113
10. requests/packages/urllib3/connectionpool.py   4,686
11. requests/sessions.py                          4,277
12. requests/utils.py                             4,041
13. requests/core.py                              3,793
14. HISTORY.rst                                   3,657
15. docs/user/advanced.rst                        3,116
16. requests/packages/chardet/langcyrillicmodel.py 3,047
17. requests/packages/chardet/big5freq.py         2,777
18. requests/packages/chardet/hebrewprober.

### Ile unikatowych autorów dotknęło plik, który zmienia się najczęściej? 
Na podstawie: 

`git log --format="===COMMIT===%nAUTHOR:%an" --name-only`

In [43]:
file_authors = defaultdict(set)
current_author = None

with open("authors.raw", encoding="utf-8") as f:
    for raw_line in f:
        line = raw_line.strip()
        if not line: continue
        
        if line == "===COMMIT===":
            current_author = None
            continue

        if line.startswith("AUTHOR:"):
            current_author = line.removeprefix("AUTHOR:")
            continue

        if current_author:
            file_authors[line].add(current_author)

file_authors = dict(sorted(
    file_authors.items(),
    key=lambda x: len(x[1]),
    reverse=True
))

print(f"Liczba pliów: {len(file_authors)}")

Liczba pliów: 461


In [47]:
file_name = "requests/models.py"
authors = file_authors[file_name]
print(f"Plik {file_name} modyfikowało {len(authors)} autorów.")

Plik requests/models.py modyfikowało 200 autorów.
